In [ ]:
# MQTT Publisher 應用程式

這個 notebook 示範如何使用 paho-mqtt 建立一個 MQTT Publisher（發布者）應用程式。


In [ ]:
import paho.mqtt.client as mqtt
import time
import json
import random

# MQTT Broker 設定（使用公開測試 Broker）
BROKER_HOST = "test.mosquitto.org"  # 公開測試 MQTT Broker
BROKER_PORT = 1883
TOPIC = "test/raspberry_pi_pico"  # 測試主題（建議使用唯一主題名稱）
CLIENT_ID = f"publisher_test_{random.randint(1000, 9999)}"  # 隨機客戶端 ID 避免衝突

print("MQTT Publisher 測試設定完成")
print(f"Broker: {BROKER_HOST}:{BROKER_PORT}")
print(f"Topic: {TOPIC}")
print(f"Client ID: {CLIENT_ID}")


In [ ]:
# 建立 MQTT 客戶端
client = mqtt.Client(client_id=CLIENT_ID)

# 連線狀態標記
connected = False

# 連線回調函數
def on_connect(client, userdata, flags, rc):
    global connected
    if rc == 0:
        connected = True
        print("✓ 成功連接到 MQTT Broker")
        print(f"  - 連線標誌: {flags}")
    else:
        connected = False
        error_messages = {
            1: "協議版本不正確",
            2: "客戶端 ID 無效",
            3: "伺服器不可用",
            4: "用戶名或密碼錯誤",
            5: "未授權"
        }
        error_msg = error_messages.get(rc, f"未知錯誤")
        print(f"✗ 連接失敗，錯誤代碼 {rc}: {error_msg}")

# 發布回調函數
def on_publish(client, userdata, mid):
    print(f"✓ 消息已發布 (Message ID: {mid})")

# 設定回調函數
client.on_connect = on_connect
client.on_publish = on_publish

# 連接到 Broker
print(f"\n正在連接到 {BROKER_HOST}:{BROKER_PORT}...")
try:
    client.connect(BROKER_HOST, BROKER_PORT, 60)
    client.loop_start()  # 開始背景執行緒處理網路流量
    time.sleep(2)  # 等待連線建立
    
    if connected:
        print("✓ 準備就緒，可以開始發布消息")
    else:
        print("⚠ 連線狀態未知，請檢查網路連線")
except Exception as e:
    print(f"✗ 連接錯誤: {e}")
    print("  請檢查：")
    print("  1. 網路連線是否正常")
    print("  2. Broker 地址是否正確")
    print("  3. 防火牆是否阻擋了連接")


In [ ]:
# 發布簡單文字消息
message = "Hello MQTT!"
result = client.publish(TOPIC, message, qos=1)
if result.rc == mqtt.MQTT_ERR_SUCCESS:
    print(f"發布消息: {message}")
else:
    print(f"發布失敗，錯誤代碼: {result.rc}")


In [ ]:
# 發布 JSON 格式消息
data = {
    "sensor": "temperature",
    "value": 25.5,
    "unit": "celsius",
    "timestamp": time.time()
}
json_message = json.dumps(data)
result = client.publish(TOPIC, json_message, qos=1)
if result.rc == mqtt.MQTT_ERR_SUCCESS:
    print(f"發布 JSON 消息: {json_message}")
else:
    print(f"發布失敗，錯誤代碼: {result.rc}")


In [ ]:
# 連續發布多條消息
for i in range(5):
    message = f"消息 #{i+1}: 當前時間 {time.strftime('%Y-%m-%d %H:%M:%S')}"
    result = client.publish(TOPIC, message, qos=1)
    if result.rc == mqtt.MQTT_ERR_SUCCESS:
        print(f"✓ {message}")
    time.sleep(1)  # 每秒發布一條消息


In [ ]:
# 關閉連線
client.loop_stop()  # 停止背景執行緒
client.disconnect()
print("✓ MQTT 連線已關閉")


## 測試說明

這個 notebook 已設定為使用公開的測試 MQTT Broker，可以直接執行測試。

### 測試步驟：
1. **執行第一個 cell**：導入套件並設定參數
2. **執行第二個 cell**：建立連線
3. **執行後續 cells**：測試發布不同類型的消息

### 測試 MQTT 消息接收

你可以使用以下方式測試消息是否成功發布：

**方法 1：使用 MQTT 客戶端工具**
- 安裝 MQTTX 或 MQTT.fx
- 連接到 `test.mosquitto.org:1883`
- 訂閱主題 `test/raspberry_pi_pico`

**方法 2：使用命令行工具**
```bash
mosquitto_sub -h test.mosquitto.org -t "test/raspberry_pi_pico"
```

**方法 3：在另一個 notebook 中建立 Subscriber**

### QoS 等級說明：
- **QoS 0**：最多一次（fire and forget，最快但不保證送達）
- **QoS 1**：至少一次（至少送達一次，可能重複，目前使用）
- **QoS 2**：僅一次（確保只送達一次，最慢但最可靠）

### 其他公開測試 Broker：
- `test.mosquitto.org` (port 1883) ← 目前使用
- `broker.hivemq.com` (port 1883)
- `mqtt.eclipseprojects.io` (port 1883)
